In [ ]:
import torch
from torch import optim
from torch import nn
import numpy as np
import mlflow  # 머신러닝 실험 기록 + 모델 관리 + 배포 도구


영화 - 조폭, 액션
[액션씬, 욕설씬]
[80,20] -> 액션
[10,90] -> 조폭


In [ ]:
mlflow.start_run()
xData = np.array([[80, 20], [95, 5], [10, 90], [90, 10], [5, 95]], dtype=np.float32) # 입력데이터
yData = np.array([[1, 0], [1, 0], [0, 1], [1, 0], [0, 1]], dtype=np.float32) # 정답(원-핫(one-hot) 벡터)

label=['액션','조폭']
xData=torch.from_numpy(xData) # PyTorch 텐서로 변환
yData=torch.from_numpy(yData)


In [ ]:
device="cpu" # 연산이 전부 CPU에서 이루어짐

In [ ]:
#모델 정의 → 손실 함수 → 옵티마이저

class MovieGenreNN(nn.Module):  # nn.Module을 상속받아 새로운 신경망 클래스를 정의
    def __init__(self):
        super().__init__()
        self.mgnn = nn.Sequential(
            nn.Linear(2 * 1, 100),  # 입력 2차원 → 은닉 100차원으로 완전연결(FC) 레이어.
                                    # 특징이 2개
            nn.ReLU(),  # 비선형 활성화 함수.
            nn.Linear(100, 50), # 100 → 50.
            nn.ReLU(), 
            nn.Linear(50, 10), # 50 → 10.
            nn.ReLU(),
            nn.Linear(10, 2) # 마지막 출력은 클래스 개수(2). 처음 특징 2개와 일치시켜야 함
        )
    def forward(self, x):
        return self.mgnn(x) # 순전파(forward) 메서드: 입력 x를 self.mgnn에 통과시켜 결과를 반환

model = MovieGenreNN().to(device) # 인공신경망 객체를 사용

lossFn = nn.CrossEntropyLoss() # 손실 함수를 지정
o = optim.Adam(model.parameters(), lr=0.0001) # Adam 옵티마이저 - lr=0.0001 (학습률)이 꽤 작은 편

In [ ]:
# 훈련 루프
epoch = 1000 # 훈련을 반복할 횟수
model.train() # 훈련 모드로 전환

for step in range(epoch):
    for i, x in enumerate(xData):
        x = x.to(device)  #현재 입력 x와 대응되는 레이블 y를 CPU → device로 이동
        y = yData[i].to(device)

        predY = model(x) # x를 모델에 통과시켜 예측(로짓) 을 얻음
        l = lossFn(predY, y) # 손실(loss)을 계산

        l.backward()  # 역전파: 손실을 각 파라미터의 그래디언트로 변환
        o.step() # 옵티마이저가 그래디언트를 사용해 파라미터를 업데이트
        o.zero_grad() # 파라미터의 그래디언트를 초기화
        print(l.item()) # 현재 손실 값을 출력
           # 데이터가 적어서 1000 epoch는 과적합 위험이 있습니다. 


In [ ]:
# 모델 평가 → 사용자 입력 받아 예측 → MLflow로 저장 및 세션 종료
model.eval() # 모델을 평가 모드로 전환

# 사용자에게 숫자를 입력받음.
# “싸움”과 “욕”은 두 개의 특징(feature) — 영화의 장르를 판단할 기준

f = float(input("액션 : "))
y = float(input("욕 : "))
newMovie = np.array([[f, y]], dtype=np.float32)
newMovie = torch.from_numpy(newMovie)
result = model(newMovie)
result = nn.Softmax()(result)
result = result.argmax().item() # 확률 벡터에서 가장 큰 값을 가진 클래스의 인덱스
print(result)
print(label[result])

mlflow.pytorch.save_model(
    pytorch_model=model,
    path="MovieGenreModel"
)

mlflow.end_run()

In [ ]:
f = float(input("액션 : "))
y = float(input("욕 : "))
newMovie = np.array([[f, y]], dtype=np.float32)
newMovie = torch.from_numpy(newMovie)
result = model(newMovie)
result = nn.Softmax()(result)
result = result.argmax().item() # 확률 벡터에서 가장 큰 값을 가진 클래스의 인덱스
print(result)
print(label[result])